In [1]:
import subprocess
import sys
import re
import os
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import seaborn as sns
import seaborn.objects as so
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp

from astropy import units as u
import numpy as np
from astropy.constants import G, sigma_sb, k_B

from scipy.interpolate import PchipInterpolator
from process_evotracks import evolution_track as track

### Planet/Impact param set up

$\Delta S = \frac{\Delta Q}{T}$, where $T$ is approximated as $T_{eq}$ and core heating is ignored

- Core cooling might prolong the inflation?

$E_{imp} = \eta M_{imp} \frac{GM_p}{R_c}$

$S-S_0 = \frac{E_{imp}}{T_{eq}}$

### Code set up and compile

In [2]:
def __pl_density_cgs(radius, mass, rad_err = None, mass_err = None):
    """
    Calculate the bulk density of a planet in cgs units (g/cm³) with error propagation.

    Parameters
    ----------
    radius : float or `~astropy.units.Quantity`
        Planetary radius. If a float is provided, it is assumed to be in Earth radii.
        If a `Quantity` is provided, it must be compatible with length units.

    mass : float or `~astropy.units.Quantity`
        Planetary mass. If a float is provided, it is assumed to be in Earth masses.
        If a `Quantity` is provided, it must be compatible with mass units.

    rad_err : float, array-like, `~astropy.units.Quantity`, or None, optional
        Uncertainty in planetary radius. If a float is provided, it is assumed to be
        in the same units as radius. If array-like of length 2, treated as
        asymmetric errors [lower, upper] and averaged for symmetric approximation.
        If None, no error propagation is performed.

    mass_err : float, array-like, `~astropy.units.Quantity`, or None, optional
        Uncertainty in planetary mass. If a float is provided, it is assumed to be
        in the same units as mass. If array-like of length 2, treated as
        asymmetric errors [lower, upper] and averaged for symmetric approximation.
        If None, no error propagation is performed.

    Returns
    -------
    density : `~astropy.units.Quantity`
        Planetary bulk density in grams per cubic centimeter (g/cm³).

    density_err : `~astropy.units.Quantity`, array, or None
        Uncertainty in planetary bulk density in g/cm³. Returns None if neither
        rad_err nor mass_err is provided. If input errors are asymmetric (length-2 arrays),
        returns asymmetric errors as array [lower_error, upper_error]. Otherwise returns
        symmetric error as scalar Quantity.

    Notes
    -----
    - If both inputs are floats, the function assumes they are in Earth units.
    - Automatically converts units to CGS (centimeters and grams) before computing.
    - Volume is computed assuming a spherical planet.
    - Error propagation uses standard uncertainty propagation formulas for
        density = mass / ((4/3) * π * radius³). For asymmetric input errors,
        the method properly propagates them to asymmetric density errors by
        considering the sign of partial derivatives.
    """
    #----Unit handling-----#
    # Handle radius units
    if not isinstance(radius, u.Quantity):
        radius = radius * u.R_earth  # assume in Earth radii

    # Handle mass units
    if not isinstance(mass, u.Quantity):
        mass = mass * u.M_earth  # assume in Earth masses

    # Handle radius error units
    if rad_err is not None:
        if not isinstance(rad_err, u.Quantity):
            # Assume same units as radius before conversion
            if not isinstance(radius, u.Quantity):
                rad_err = rad_err * u.R_earth
            else:
                rad_err = rad_err * radius.unit

    # Handle mass error units
    if mass_err is not None:
        if not isinstance(mass_err, u.Quantity):
            # Assume same units as mass before conversion
            if not isinstance(mass, u.Quantity):
                mass_err = mass_err * u.M_earth
            else:
                mass_err = mass_err * mass.unit

    # Convert to cgs
    radius_cgs = radius.to(u.cm)
    mass_cgs = mass.to(u.g)
    mass_err_cgs = None
    rad_err_cgs = None
    if rad_err is not None:
        rad_err_cgs = rad_err.to(u.cm)
    if mass_err is not None:
        mass_err_cgs = mass_err.to(u.g)

    # Calculate density
    V = (4 / 3) * np.pi * radius_cgs ** 3
    density = mass_cgs / V

    # Calculate error if requested
    density_err = None
    if rad_err is not None or mass_err is not None:
        # Partial derivatives for error propagation
        # ρ = M / ((4/3)πR³)
        # ∂ρ/∂M = 1 / ((4/3)πR³) = ρ/M
        # ∂ρ/∂R = -3M / ((4/3)πR⁴) = -3ρ/R

        # Check if we have asymmetric errors
        mass_is_asymmetric = mass_err is not None and hasattr(mass_err_cgs, '__len__') and len(mass_err_cgs) == 2
        rad_is_asymmetric = rad_err is not None and hasattr(rad_err_cgs, '__len__') and len(rad_err_cgs) == 2

        if mass_is_asymmetric or rad_is_asymmetric:
            # Handle asymmetric error propagation
            dρ_dM = density / mass_cgs if mass_err is not None else 0
            dρ_dR = -3 * density / radius_cgs if rad_err is not None else 0

            # Calculate lower and upper bounds
            err_lower_terms = []
            err_upper_terms = []

            if mass_err is not None:
                if mass_is_asymmetric:
                    # For mass: positive derivative, so lower mass error -> lower density error
                    err_lower_terms.append((dρ_dM * mass_err_cgs[0]) ** 2)
                    err_upper_terms.append((dρ_dM * mass_err_cgs[1]) ** 2)
                else:
                    # Symmetric mass error
                    mass_term = (dρ_dM * mass_err_cgs) ** 2
                    err_lower_terms.append(mass_term)
                    err_upper_terms.append(mass_term)

            if rad_err is not None:
                if rad_is_asymmetric:
                    # For radius: negative derivative, so lower radius error -> upper density error
                    err_lower_terms.append((dρ_dR * rad_err_cgs[1]) ** 2)  # Note: switched indices
                    err_upper_terms.append((dρ_dR * rad_err_cgs[0]) ** 2)  # Note: switched indices
                else:
                    # Symmetric radius error
                    rad_term = (dρ_dR * rad_err_cgs) ** 2
                    err_lower_terms.append(rad_term)
                    err_upper_terms.append(rad_term)

            density_err_lower = np.sqrt(sum(err_lower_terms))
            density_err_upper = np.sqrt(sum(err_upper_terms))

            # Return as array [lower, upper] with proper units
            density_err = np.array([density_err_lower.value, density_err_upper.value]) * density.unit

        else:
            # Handle symmetric error propagation
            err_terms = []

            if mass_err is not None:
                dρ_dM = density / mass_cgs
                err_terms.append((dρ_dM * mass_err_cgs) ** 2)

            if rad_err is not None:
                dρ_dR = -3 * density / radius_cgs
                err_terms.append((dρ_dR * rad_err_cgs) ** 2)

            if err_terms:
                density_err = np.sqrt(sum(err_terms))

    if density_err is None:
        return density
    else:
        return density, density_err
    

In [3]:

E_imp = 0




runswitch = True
compilation = True #Re-compile every time
AMF = 0.3

controlEvoFile = f"controls/evo-control-f{str(AMF).replace(".","")}.dat"

structFileName = f"sim_results/impact-fenv{str(AMF).replace(".","")}-struct.dat"
evolveFileName = f"sim_results/impact-fenv{str(AMF).replace(".","")}-evo.dat"

prefix_005 = "./PlanetSolver -cn 1 0.0162495 -en 2 0.6412495 -a 9.50 30 0.05 -s 1 1 1 0.3 -m 8.0 -evolve 1"
prefix_01 = "./PlanetSolver -cn 1 0.292495 -en 2 0.607495 -a 8.9 30 0.09999 -s 1 1 1 0.3 -m 8.0 -evolve 1"
prefix_02 = "./PlanetSolver -cn 1 0.2599 -en 2 0.53995 -a 8.38 30 0.2 -s 1 1 1 0.3 -m 8.0 -evolve 1"
prefix_03 = "./PlanetSolver -cn 1 0.227495 -en 2 0.47249  -a 7.96 30 0.3 -s 1 1 1 0.3 -m 8.0 -evolve 1"
prefix_dict = {0.1:prefix_01, 0.2:prefix_02, 0.3:prefix_03}
runswitch = True
compilation = True #Re-compile every time

In [4]:
if runswitch and compilation:
    result = subprocess.run(
    "make clean", 
    shell=True, 
    capture_output=True, 
    text=True,
    timeout=300
    )
    if result.returncode != 0:
        raise OSError("OS could not handle make clean")
    print("Cleaned")

    result = subprocess.run(
    "make", 
    shell=True, 
    capture_output=True, 
    text=True,
    timeout=300
    )
    if result.returncode != 0:
        print(result.stdout)
        raise OSError(f"Could not make:{result.stderr}")
    print("Made")




Cleaned
Made


In [5]:
#multiprocessing
debug = False
def run_sims(entropy):
    entropy_str = f"{entropy:.3f}"
    #prefix_01 = "./PlanetSolver -cn 1 0.292495 -en 2 0.607495 -a 8.9 30 0.09999 -s 1 1 1 0.3 -m 8.0 -evolve 1"
    #prefix_02 = "./PlanetSolver -cn 1 0.2599 -en 2 0.53995 -a 8.38 30 0.2 -s 1 1 1 0.3 -m 8.0 -evolve 1"
    filestr = f"init_impact/init-impact-{AMF}-{entropy_str}.dat"
    runcmd = f"./PlanetSolver -cn 1 0.227495 -en 2 0.47249  -a {entropy_str} 30 0.3 -s 1 1 1 0.3 -m 8.0 -evolve 1 -wr {filestr}"
 
    try:
            
            result = subprocess.run(
            runcmd, 
            shell=True, 
            capture_output=True, 
            text=True,
            timeout=900
            )
        
            if result.returncode != 0:
                print(result.stdout)
                raise OSError("Runtime error")
            print(f"Run Success: {entropy_str}, {AMF} -> {filestr}\n")

            
            return result
    except subprocess.TimeoutExpired as timeout:
        print(f"Timeout: {entropy_str}, {AMF} -> {filestr}\n")
        return None

entropies = np.linspace(7.0, 11.0, 40)
if runswitch:
    with mp.Pool(processes=mp.cpu_count()-1) as pool:
        all_results = pool.map(run_sims, entropies)



Run Success: 7.000, 0.3 -> init_impact/init-impact-0.3-7.000.dat

Run Success: 7.205, 0.3 -> init_impact/init-impact-0.3-7.205.dat

Run Success: 7.410, 0.3 -> init_impact/init-impact-0.3-7.410.dat

Run Success: 7.821, 0.3 -> init_impact/init-impact-0.3-7.821.dat

Run Success: 8.231, 0.3 -> init_impact/init-impact-0.3-8.231.dat

Run Success: 7.103, 0.3 -> init_impact/init-impact-0.3-7.103.dat

Run Success: 7.308, 0.3 -> init_impact/init-impact-0.3-7.308.dat

Run Success: 7.513, 0.3 -> init_impact/init-impact-0.3-7.513.dat

Run Success: 7.923, 0.3 -> init_impact/init-impact-0.3-7.923.dat

Timeout: 7.615, 0.3 -> init_impact/init-impact-0.3-7.615.dat
Timeout: 8.026, 0.3 -> init_impact/init-impact-0.3-8.026.dat


Run Success: 8.641, 0.3 -> init_impact/init-impact-0.3-8.641.dat

Run Success: 7.718, 0.3 -> init_impact/init-impact-0.3-7.718.dat

Run Success: 8.128, 0.3 -> init_impact/init-impact-0.3-8.128.dat

Timeout: 8.333, 0.3 -> init_impact/init-impact-0.3-8.333.dat

Timeout: 8.436, 0.3 ->

In [6]:
if runswitch:
    for result in all_results:
        if result is not None:
            print(result.stdout)
            print(result.stderr)
        else:
            print("None")

4.144028 2.641872e+01 2.813503e+01 166.218167 7.566752


4.100110 2.640877e+01 2.810477e+01 166.285522 7.496276


4.758044 2.639986e+01 2.807764e+01 254.572485 7.604058


4.774400 2.639198e+01 2.805369e+01 262.535971 7.554498


3.988809 2.638395e+01 2.802930e+01 166.398394 7.326989


3.948218 2.637419e+01 2.799963e+01 166.903253 7.264739


None
3.828018 2.634790e+01 2.791986e+01 166.582083 7.100367


3.755580 2.633155e+01 2.787029e+01 166.794735 7.004952


3.675279 2.631441e+01 2.781838e+01 166.478907 6.906008


None
3.524690 2.628076e+01 2.771662e+01 166.633764 6.729424


3.448454 2.626345e+01 2.766434e+01 166.789534 6.645568


None
None
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
Planet integration failed.
3.251185 2.620568e+01 2.74902